# Tutorial: from a published SAE to a spatial report (CPU)

This notebook runs the spatial stages of the interpreTFM pipeline on a small crop of the public CosMx human lung cancer section (NSCLC, section 8; about 2,800 cells of a 4,500 × 4,500 px = 810 × 810 µm window), starting from the scGPT `layer_1.norm2` sparse autoencoder (SAE) of the manuscript and the retained features already projected to these cells. No GPU and no foundation-model weights are needed; everything runs on a CPU in a few minutes. It shows

1. what the SAE export contains and how it is loaded (`interpretfm.hub`);
2. what the `cells` stage computes (features projected to cells), checked on a slice of the activation store;
3. the stages `niches → interfaces → lr_screen → report` from one configuration file, and how to read the report.

A last, optional section shows how activations are extracted from scGPT on a GPU.

The data come as one bundle from either a local folder or the Hugging Face repository of the release (`SOURCE` below). On Colab, run the next cell first to install the package.

In [ ]:
import importlib.util, subprocess, sys

ON_COLAB = importlib.util.find_spec("google.colab") is not None
if ON_COLAB:  # a plain CPU install: PyTorch from the CPU index, then the package with the hub extra
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "torch", "--index-url", "https://download.pytorch.org/whl/cpu"], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "interpretfm[hub] @ git+https://github.com/OliiverHu/interpreTFM.git"], check=True)

## 1. The data bundle and the SAE

`interpretfm.hub.tutorial_bundle` returns the root folder of the bundle. With `SOURCE = "local"` it is the folder `BUNDLE` (the bundle of the release, downloaded by hand or built from the run directories); with `SOURCE = "hub"` the files are fetched from the Hugging Face repository `REPO_ID` (the placeholder is replaced by the published repository id).

In [ ]:
import os
from pathlib import Path

import anndata as ad
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import yaml

from interpretfm import hub

SOURCE = "local"                                       # "local" or "hub"
BUNDLE = os.environ.get("INTERPRETFM_TUTORIAL_BUNDLE", "interpretfm_tutorial_bundle")
REPO_ID = hub.DEFAULT_REPO                             # "<hf-user>/interpretfm-v1" until the release is published
MODEL, LAYER, DATASET = "scgpt", "layer_1.norm2", "cosmx"

root = hub.tutorial_bundle(SOURCE, path=BUNDLE, repo_id=REPO_ID)
sorted(p.relative_to(root).as_posix() for p in root.rglob("*") if p.is_file())

In [ ]:
crop = ad.read_h5ad(root / "tutorial" / "cosmx_crop.h5ad")
print(crop)
print("raw counts:", crop.X[:3].sum(1).A.ravel(), "| µm per pixel:", crop.uns["interpretfm_tutorial"]["um_per_unit"])
crop.obs["author_cell_type"].value_counts().head(8)

In [ ]:
xy = crop.obsm["spatial"] * 0.18
types = crop.obs["author_cell_type"].astype(str)
top = types.value_counts().index[:8]
fig, ax = plt.subplots(figsize=(5.2, 4.6), dpi=100)
for t in top:
    m = (types == t).values
    ax.scatter(xy[m, 0], xy[m, 1], s=4, label=f"{t} ({m.sum()})")
ax.set_aspect("equal"); ax.set_xlabel("x (µm)"); ax.set_ylabel("y (µm)"); ax.set_title("CosMx crop, author cell types")
ax.legend(fontsize=7, markerscale=3, loc="center left", bbox_to_anchor=(1.0, 0.5), frameon=False)
plt.show()

### The exported SAE

`hub.load_sae` returns a checkpoint dict (`state_dict`, `d_in`, `n_latents`, `input_scale`, the training `spec`) that `interpretfm.sae.Encoder` accepts; the folder also holds the retained feature–concept table and the metrics of the layer beside their baselines.

In [ ]:
from interpretfm.sae import Encoder

ckpt = hub.load_sae(root, MODEL, LAYER, dataset=DATASET)
encoder = Encoder(ckpt)
print(f"{ckpt['architecture']}: d_in {ckpt['d_in']}, n_latents {ckpt['n_latents']}, input_scale {ckpt['input_scale']:.4f}, spec {ckpt['spec']}")

metrics = pd.read_json(hub.layer_dir(root, MODEL, LAYER, dataset=DATASET) / "metrics.json", typ="series")
f1 = pd.DataFrame(metrics["concept_f1"]).T[["unit", "representation", "n_concepts", "concept_f1"]]
f1.pivot(index="unit", columns="representation", values="concept_f1").round(3)

In [ ]:
features = hub.load_features(root, MODEL, LAYER, dataset=DATASET)
print(len(features), "retained features;", features["concept"].nunique(), "concepts; mean test F1", round(features["test_f1"].mean(), 3))
features.sort_values("test_f1", ascending=False).head(8)[["feature", "concept", "term_name", "n_panel_genes", "valid_f1", "test_f1"]]

## 2. What the `cells` stage computes

The `cells` stage reads the activation store (one row of hidden states per gene token of every cell), encodes every row with the SAE, and averages the retained features over the gene tokens of each cell. The bundle holds one shard of the store restricted to 80 cells of the crop, and `tutorial/cells_layer_1.norm2.h5ad` holds the features of all crop cells as the manuscript run projected them over the whole section. The same function the stage uses reproduces those values.

In [ ]:
from interpretfm.cells import accumulate_cell_profiles

store = root / "tutorial" / "store"
cells_pub = ad.read_h5ad(root / "tutorial" / "cells_layer_1.norm2.h5ad")
fidx = cells_pub.var["feature"].astype(int).tolist()
prof = accumulate_cell_profiles(store, LAYER, encoder, [str(o) for o in crop.obs_names], special_tokens=["60694", "60695"],
                                feature_idx=fidx, batch_size=4096)
ours = pd.DataFrame(prof.sae, index=crop.obs_names[prof.cell_index], columns=cells_pub.var_names)
theirs = cells_pub[ours.index].to_df()
print(f"{len(ours)} cells, {ours.shape[1]} features; gene tokens per cell: median {int(np.median(prof.counts))};",
      f"max |difference| to the published projection: {np.abs(ours.values - theirs.values).max():.2e}")
ours.iloc[:4, :5].round(3)

## 3. Niches, interfaces, ligand–receptor screen and the report

One YAML configuration drives the pipeline. Here the projected features replace the model-level stages: `cells_h5ad` points `niches` and `interfaces` at the published features (so `extract`, `sae` and `cells` need not run), and `report: {spatial_only: true}` builds the report from the spatial stages alone. The settings follow the manuscript's CosMx configuration (radii in µm, three niches at 18 µm, the calibrated interface test on the author labels grouped as in the paper, the screen at 18 µm) with fewer permutations and seeds so that the notebook runs in minutes. Paths are relative to the configuration file.

In [ ]:
run_root = Path("tutorial_run")
run_root.mkdir(exist_ok=True)
rel = lambda p: os.path.relpath(p, run_root)
config = {
    "run": {"name": "cosmx_crop", "out_dir": "runs/cosmx_crop", "seed": 0},
    "dataset": {"h5ad": rel(root / "tutorial/cosmx_crop.h5ad"), "counts_layer": None,
                "gene_id": {"column": "index", "kind": "ensembl"},
                "labels": {"column": "author_cell_type", "groups": rel(root / "tutorial/cosmx_author_groups.csv")},
                "spatial": {"obsm": "spatial", "um_per_unit": 0.18}, "qc": {"min_transcripts": 0, "min_genes": 0}},
    "model": {"name": MODEL, "checkpoint": None, "env": {}, "layers": [LAYER], "representative_layer": LAYER,
              "cell_pooling": {"kind": "token", "token": "60695"}, "special_tokens": ["60694", "60695"]},
    "stages": {
        "niches": {"cells_h5ad": {LAYER: rel(root / "tutorial/cells_layer_1.norm2.h5ad")},
                   "radii_um": [18.0, 21.6, 27.0], "k": [3, 4, 5], "methods": ["kmeans", "gmm"], "spaces": ["m", "xm"],
                   "seeds": [0, 1], "min_cluster": 50, "n_pcs": 50, "n_null": 5,
                   "baselines": ["expression_svd50", "label_composition"],
                   "label_composition": {"radius_um": 21.6, "k": 3, "n_pcs": 15, "n_init": 10, "seed": 0},
                   "seed_stability": {"k": [4, 5]}, "concepts": {"top_k": 25, "min_cells": 20},
                   "fixed_configs": [{"radius_um": 18.0, "space": "xm", "method": "kmeans", "k": 3}]},
        "interfaces": {"cells_h5ad": {LAYER: rel(root / "tutorial/cells_layer_1.norm2.h5ad")},
                       "edge_radius_um": 21.6, "block_um": 21.6, "legacy_tile_um": 72.0, "length_scales_um": [10.8, 27.0, 72.0],
                       "n_perm": 50, "calib_perm": 20, "calib_reps": 2, "legacy_perm": 10, "min_edges": 20,
                       "exclude_groups": ["low_quality", "unresolved"], "covariates": {"area": {"product": ["Width", "Height"]}},
                       "niche": {"config": "r18um/xm/kmeans/k3"}, "markers": {"enabled": True, "top_n": 10},
                       "fired_on": {"enabled": False}},
        "lr_screen": {"radius_um": 18.0, "thresholds": [1, 3], "n_perm": 100, "seed": 7, "min_positive": 30,
                      "q_max": 0.05, "fold_min": 1.2, "niche": {"config": "r18um/xm/kmeans/k3"}},
        "report": {"spatial_only": True},
    },
    "output": {"html": True, "json": True},
}
cfg_path = run_root / "cosmx_crop.yaml"
cfg_path.write_text(yaml.safe_dump(config, sort_keys=False))
print(cfg_path.read_text()[:600], "...")

`interpretfm validate` checks the configuration and the files it names; `interpretfm run --stages ...` runs the stages in order, caching each one in the run directory (a second call skips finished stages). The same commands work from a shell: `interpretfm validate tutorial_run/cosmx_crop.yaml` and `interpretfm run tutorial_run/cosmx_crop.yaml --stages ingest,niches,interfaces,lr_screen,report`.

In [ ]:
import logging, time

from interpretfm.cli import main

logging.getLogger().handlers.clear()
assert main(["validate", str(cfg_path)]) == 0
t0 = time.time()
assert main(["run", str(cfg_path), "--stages", "ingest,niches,interfaces,lr_screen,report"]) == 0
print(f"finished in {time.time() - t0:.0f} s")
main(["status", str(run_root / "runs/cosmx_crop")])

### Reading the report

`13_report/report.json` holds every number beside its baseline; `13_report/report.html` is the same as one self-contained page. Each section carries a note on what it measures and what it does not (`section_notes`).

In [ ]:
import json

run_dir = run_root / "runs/cosmx_crop"
report = json.load(open(run_dir / "13_report/report.json"))
for section in ("niches", "interfaces", "lr_screen"):
    print(f"[{section}] {report['section_notes'][section]}\n")

**Niches.** The sweep over radius, clustering method, feature space (`m` = features, `xm` = features with coordinates) and `k` is scored by seed stability (ARI between seeds), purity against a tile-shuffle null and separability; the selected configuration and the manuscript's fixed configuration are both validated. Read the model's ARI/NMI against the author labels next to the expression-only partition (SVD of the counts, same sweep) and the label-composition partition (neighbourhood cell-type composition): a niche partition from SAE features is informative when it tells the labels something the expression-only partition does not, at a seed stability the baselines share.

In [ ]:
n = report["niches"][LAYER]
rows = {"SAE features (" + n["config"] + ")": n["model"]}
for name in ("expression_winner", "expression_at_model_config", "label_composition"):
    b = n["baselines"].get(name)
    if b:
        rows[name + (" (" + b["config"] + ")" if b.get("config") else "")] = b
pd.DataFrame(rows).T[["ari", "nmi", "seed_ari", "purity", "purity_null_mean", "purity_z"]].astype(float).round(3)

In [ ]:
labels = pd.read_csv(run_dir / "10_niches" / LAYER / "labels.csv", dtype={"cell": str}).set_index("cell")
col = [c for c in labels.columns if c.startswith("niche_at_")][0]
niche = labels[col].reindex(crop.obs_names)
top_concepts = pd.read_csv(run_dir / "10_niches" / LAYER / "top_concepts.csv")
fig, axes = plt.subplots(1, 2, figsize=(9.5, 4.2), dpi=100, gridspec_kw={"width_ratios": [1.1, 1]})
for k in sorted(niche.dropna().unique()):
    m = (niche == k).fillna(False).to_numpy(dtype=bool)
    axes[0].scatter(xy[m, 0], xy[m, 1], s=4, label=f"niche {k} ({m.sum()} cells)")
axes[0].set_aspect("equal"); axes[0].set_title(f"niches at {col.replace('niche_at_', '')}"); axes[0].set_xlabel("x (µm)"); axes[0].set_ylabel("y (µm)")
axes[0].legend(fontsize=7, markerscale=3, frameon=False)
ct = pd.crosstab(niche.dropna().astype(int), types[niche.notna()].values, normalize="index")
ct[ct.sum().sort_values(ascending=False).index[:6]].plot.barh(ax=axes[1], stacked=True, width=0.7, fontsize=8)
axes[1].set_xlabel("share of cells"); axes[1].set_ylabel("niche"); axes[1].legend(fontsize=7, frameon=False, loc="lower right"); axes[1].set_title("author cell types per niche")
fig.tight_layout()
display(fig)
plt.close(fig)

The niche concept labels (`top_concepts.csv`) are the retained concepts whose feature scores are highest inside each niche relative to the rest (Wilcoxon, BH-adjusted), ranked by effect; they describe what the SAE features say about each niche.

In [ ]:
top_concepts.groupby("niche").head(3)[["niche", "concept", "term_name", "effect", "qval"]].round(3)

**Interfaces.** For every pair of cell groups with enough contact edges, the program score (the mean of the retained features, adjusted within each label for transcript counts, detected genes and cell area) at the interface is compared with a permutation null inside spatial blocks, niches and labels. The share of pairs with |z| > 2 under the legacy tile null is reported beside the share under the calibrated null; the call threshold `z*` is set on contact-free smooth random fields with the same neighbour correlation, and the false-positive rate on those fields is the calibration check. On a crop this small most pairs have few edges, so read the counts, not the identities of the pairs.

In [ ]:
i = report["interfaces"][LAYER]
print(f"{i['n_cells']} cells, {i['n_edges']} contact edges at {i['units']['edge_radius']} µm; {i['n_niches']} niches ({i['niche_column']})")
print("pairs tested (niche-stratified / niche-free):", i["n_pairs"])
print("share |z| > 2: legacy tile null", round(i["share_abs_z_gt2"]["legacy_tile"], 3), "| calibrated", round(i["share_abs_z_gt2"]["calibrated"], 3))
print("z*:", {k: round(v, 2) for k, v in i["z_star"].items()}, "| false-positive rate on smooth fields:", {k: round(v, 3) for k, v in i["fpr"].items()})
print("calls:", i["calls"])
pairs = pd.read_csv(run_dir / "11_interfaces" / LAYER / "niche_pairs.csv")
pairs.sort_values("z_v2", key=np.abs, ascending=False).head(6)[["group_a", "group_b", "niche", "n_edges", "z_v1", "z_v2", "effect_pct_v2", "call_v2"]].round(2)

**Ligand–receptor screen.** The screen is model-free: for every CellChatDB pair on the panel it counts ligand-positive cells with a receptor-positive neighbour within the radius against a within-cell-type permutation null (fold, z, BH q), at two positivity thresholds (≥ 1 and ≥ 3 transcripts); a *robust* hit passes at both. The placement table puts each hit in the niches (share of close pairs in a niche over the niche's share of cells; a ratio above 1 means the pair is concentrated there). Read it as spatial association of expression, not as signalling.

In [ ]:
s = report["lr_screen"]
print(f"radius {s['radius_um']} µm = {s['radius_native']} px, {s['mean_other_cells']:.2f} other cells within the radius on average; "
      f"{s['n_pairs_on_panel']} CellChatDB pairs on the panel; thresholds {s['thresholds']}")
print(f"tested pairs {s['tested']} | hits {s['hits']} (primary threshold {s['primary']}) | robust {s['n_robust']} {s['robust']} | depleted {s['n_depleted']}")
screen = pd.read_csv(run_dir / "12_lr_screen/screen.csv")
screen[screen[["hit_thr1", "hit_thr3"]].any(axis=1)][["interaction_name", "obs_thr1", "fold_thr1", "z_thr1", "q_thr1", "hit_thr1", "fold_thr3", "z_thr3", "hit_thr3", "robust_hit"]].head(8).round(3)

In [ ]:
placement = run_dir / "12_lr_screen" / LAYER / "placement.csv"
pd.read_csv(placement).pivot_table(index="interaction_name", columns="niche", values="ratio").round(2) if placement.exists() else "no hit to place on the crop"

The HTML page (`tutorial_run/runs/cosmx_crop/13_report/report.html`) shows the same tables with figures; open it in a browser. Every section states its baselines first: the expression-only and label-composition partitions for niches, the legacy tile null and the smooth-field false-positive rate for interfaces, the permutation null for the screen.

## 4. Optional: extracting activations with scGPT on a GPU

The model-level stages start from an activation store. To build one, the `extract` stage runs the model's adapter in the model's own environment (`model.env`) as a child process and writes the sharded store into the run directory. The configuration below (the manuscript's `configs/paper/extract_cosmx_scgpt.yaml` restricted to the crop) needs the public scGPT whole-human checkpoint folder (`best_model.pt`, `vocab.json`) and the `interpretfm` conda environment of `envs/interpretfm.yml` (with `nnsight` installed as described in the installation guide); set `RUN_GPU = True` on a machine with both.

In [ ]:
RUN_GPU = False
SCGPT_CHECKPOINT = os.environ.get("INTERPRETFM_SCGPT_CHECKPOINT", "/path/to/scGPT/whole-human-pretrain")

extract_config = {
    **{k: v for k, v in config.items() if k in ("dataset",)},
    "run": {"name": "cosmx_crop_extract", "out_dir": "runs/cosmx_crop_extract", "seed": 0},
    "model": {**config["model"], "checkpoint": SCGPT_CHECKPOINT, "env": {"conda": "interpretfm"}},  # or {} to run in this interpreter
    "stages": {"extract": {"batch_size": 8, "max_length": 512, "dtype": "float32", "rows_per_shard": 50000, "device": "cuda",
                           "gene_keys": {"symbol_column": "feature_name"}}},
    "output": {"html": False, "json": True, "keep_store": True},
}
extract_path = run_root / "cosmx_crop_extract.yaml"
extract_path.write_text(yaml.safe_dump(extract_config, sort_keys=False))
if RUN_GPU:
    assert main(["run", str(extract_path), "--stages", "ingest,extract"]) == 0
    # then the model-level stages on the new store, e.g. the published SAE through stages.sae.checkpoints:
    # interpretfm run tutorial_run/cosmx_crop_extract.yaml --stages tis,sae,concepts,concept_eval,gene_readout,cells,layers,niches,interfaces,lr_screen,report
else:
    print("skipped (RUN_GPU = False); configuration written to", extract_path)

After extraction, `02_extract/<layer>/manifest.json` records the rows, cells and tokens written and the child's interpreter and library versions, and `interpretfm.extract.compare_stores` compares a new store with a cached one cell by cell (the reproduction tests use it to check the manuscript's stores).